# Claims Agent Test Notebook

This notebook tests the `smol_claims_agent.py` implementation, including:
- Direct tool function testing
- Agent interaction testing
- Error handling
- Full workflow testing


In [ ]:
# Setup: Import dependencies and load environment
import os
import sys
import json
from pathlib import Path
from dotenv import load_dotenv

# Add project root to Python path (works in Jupyter notebooks)
# Strategy: Check current directory and parent directory
current_dir = Path.cwd().resolve()

# If we're in src/, go up one level
if current_dir.name == 'src':
    project_root = current_dir.parent
elif (current_dir / 'src').exists():
    # We're already at project root
    project_root = current_dir
else:
    # Try going up one level
    project_root = current_dir.parent
    if not (project_root / 'src').exists():
        project_root = current_dir

# Add to path if not already there
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
    print(f"✓ Added project root to path: {project_root}")
else:
    print(f"✓ Project root already in path: {project_root}")

# Load environment variables (try both project root and current dir)
env_file = project_root / '.env'
if env_file.exists():
    load_dotenv(env_file, override=True)
else:
    load_dotenv(override=True)

# Verify API key is set
api_key = os.getenv('OPENAI_API_KEY')
if api_key:
    print(f"✓ OpenAI API Key loaded (starts with: {api_key[:8]}...)")
else:
    print("✗ WARNING: OPENAI_API_KEY not set in environment")


✓ OpenAI API Key loaded (starts with: sk-proj-...)


In [ ]:
# Import the agent and tools
from src.agents.smol_claims_agent import claims_smol_agent
from src.tools import claims_tools as ct

print("✓ Claims agent and tools imported successfully")
print(f"✓ Agent name: {claims_smol_agent.name}")
print(f"✓ Number of tools: {len(claims_smol_agent.tools)}")


✓ Claims agent and tools imported successfully
✓ Agent name: claims_specialist
✓ Number of tools: 5


## 1. Direct Tool Testing

Test the underlying tool functions directly before testing the agent.


In [9]:
# Test 1: List claims (should be empty initially)
employee_id = "mark_tan"
result = ct.list_claims(employee_id)
print("List Claims Result:")
print(json.dumps(json.loads(result), indent=2))


List Claims Result:
{
  "employee_id": "mark_tan",
  "claims": []
}


In [10]:
# Test 2: Create a draft claim
draft_result = ct.draft_medical_claim(
    employee_id=employee_id,
    medical_provider="ABC Medical Clinic",
    receipt_no="R-12345",
    receipt_date="2025-01-15",
    receipt_amount=85.50,
    diagnosis="Common cold",
    gst_inclusive=True
)

draft_data = json.loads(draft_result)
print("Draft Claim Result:")
print(json.dumps(draft_data, indent=2))

# Extract claim_id for later tests
if "claim_id" in draft_data:
    claim_id = draft_data["claim_id"]
    print(f"\n✓ Draft created with claim_id: {claim_id}")
else:
    print("\n✗ Error: No claim_id in response")
    claim_id = None


Draft Claim Result:
{
  "status": "draft",
  "message": "Draft created successfully.",
  "claim_id": "claim-95302",
  "details": {
    "medical_provider": "ABC Medical Clinic",
    "receipt_no": "R-12345",
    "receipt_date": "2025-01-15",
    "receipt_amount": 85.5,
    "diagnosis": "Common cold",
    "gst_inclusive": true
  }
}

✓ Draft created with claim_id: claim-95302


In [11]:
# Test 3: Verify the claim appears in list
if claim_id:
    result = ct.list_claims(employee_id)
    claims_data = json.loads(result)
    print(f"Total claims for {employee_id}: {len(claims_data.get('claims', []))}")
    print("\nClaims list:")
    print(json.dumps(claims_data, indent=2))


Total claims for mark_tan: 1

Claims list:
{
  "employee_id": "mark_tan",
  "claims": [
    {
      "claim_id": "claim-95302",
      "status": "draft",
      "medical_provider": "ABC Medical Clinic",
      "receipt_no": "R-12345",
      "receipt_date": "2025-01-15",
      "receipt_amount": 85.5,
      "diagnosis": "Common cold",
      "gst_inclusive": true
    }
  ]
}


In [12]:
# Test 4: Update the draft claim (correct the amount)
if claim_id:
    update_result = ct.update_medical_claim(
        employee_id=employee_id,
        claim_id=claim_id,
        field="receipt_amount",
        value="95.50"  # Corrected amount
    )
    update_data = json.loads(update_result)
    print("Update Claim Result:")
    print(json.dumps(update_data, indent=2))


Update Claim Result:
{
  "status": "updated",
  "message": "Updated receipt_amount to 95.50 for claim claim-95302.",
  "current_data": {
    "medical_provider": "ABC Medical Clinic",
    "receipt_no": "R-12345",
    "receipt_date": "2025-01-15",
    "receipt_amount": 95.5,
    "diagnosis": "Common cold",
    "gst_inclusive": true
  }
}


In [13]:
# Test 5: Update another field (diagnosis)
if claim_id:
    update_result = ct.update_medical_claim(
        employee_id=employee_id,
        claim_id=claim_id,
        field="diagnosis",
        value="Sinus infection"
    )
    update_data = json.loads(update_result)
    print("Update Diagnosis Result:")
    print(json.dumps(update_data, indent=2))


Update Diagnosis Result:
{
  "status": "updated",
  "message": "Updated diagnosis to Sinus infection for claim claim-95302.",
  "current_data": {
    "medical_provider": "ABC Medical Clinic",
    "receipt_no": "R-12345",
    "receipt_date": "2025-01-15",
    "receipt_amount": 95.5,
    "diagnosis": "Sinus infection",
    "gst_inclusive": true
  }
}


In [14]:
# Test 6: Test error cases - Update non-existent claim
error_result = ct.update_medical_claim(
    employee_id=employee_id,
    claim_id="claim-99999",
    field="receipt_amount",
    value="100.00"
)
print("Error Test (non-existent claim):")
print(json.loads(error_result))


Error Test (non-existent claim):
{'error': 'Claim not found.'}


In [15]:
# Test 7: Test error cases - Invalid date format
error_result = ct.draft_medical_claim(
    employee_id=employee_id,
    medical_provider="Test Clinic",
    receipt_no="R-99999",
    receipt_date="2025/01/15",  # Wrong format
    receipt_amount=50.00,
    diagnosis="Test",
    gst_inclusive=False
)
print("Error Test (invalid date format):")
print(json.loads(error_result))


Error Test (invalid date format):
{'error': 'Invalid date format. Use YYYY-MM-DD.'}


In [16]:
# Test 8: Submit the claim
if claim_id:
    submit_result = ct.submit_medical_claim(employee_id, claim_id)
    submit_data = json.loads(submit_result)
    print("Submit Claim Result:")
    print(json.dumps(submit_data, indent=2))


--- SYSTEM: Claim claim-95302 formally submitted to backend ---
Submit Claim Result:
{
  "status": "submitted",
  "message": "Claim claim-95302 has been submitted for processing.",
  "employee_id": "mark_tan"
}


In [17]:
# Test 9: Try to update a submitted claim (should fail)
if claim_id:
    error_result = ct.update_medical_claim(
        employee_id=employee_id,
        claim_id=claim_id,
        field="receipt_amount",
        value="100.00"
    )
    print("Error Test (update submitted claim):")
    print(json.loads(error_result))


Error Test (update submitted claim):
{'error': 'Cannot update claim. Current status is submitted.'}


In [18]:
# Test 10: Try to submit already submitted claim (should fail)
if claim_id:
    error_result = ct.submit_medical_claim(employee_id, claim_id)
    print("Error Test (resubmit claim):")
    print(json.loads(error_result))


Error Test (resubmit claim):
{'error': 'Claim is already submitted.'}


## 2. Agent Interaction Testing

Test the agent's ability to handle natural language queries and use tools appropriately.


In [19]:
# Test 11: Agent - List claims
response = claims_smol_agent.run("Can you show me all my medical claims?")
print("Agent Response:")
print(response)
print("\n" + "="*80)


╭────────────────────────────────────────── New run - claims_specialist ──────────────────────────────────────────╮
│                                                                                                                 │
│ Can you show me all my medical claims?                                                                          │
│                                                                                                                 │
╰─ OpenAIModel - gpt-4.1-mini ────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'list_claims_tool' with arguments: {'employee_id': 'mark_tan'}                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: {
  "employee_id": "mark_tan",
  "claims": |
    {
      "claim_id": "claim-95302",
      "status": "submitted",
      "medical_provider": "ABC Medical Clinic",
      "receipt_no": "R-12345",
      "receipt_date": "2025-01-15",
      "receipt_amount": 95.5,
      "diagnosis": "Sinus infection",
      "gst_inclusive": true
    }
  ]
}

[Step 1: Duration 1.90 seconds| Input tokens: 2,299 | Output tokens: 19]

...

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'final_answer' with arguments: {'answer': 'Here are your medical claims:\n\nClaim ID:             │
│ claim-95302\nStatus: submitted\nMedical Provider: ABC Medical Clinic\nReceipt No: R-12345\nReceipt Date:        │
│ 2025-01-15\nReceipt Amount: 95.50\nDiagnosis: Sinus infection\nGST Inclusive: Yes\n\nIf you want to do anything │
│ else with these claims, please let me know.'}                                                                   │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: Here are your medical claims:

Claim ID: claim-95302
Status: submitted
Medical Provider: ABC Medical Clinic
Receipt No: R-12345
Receipt Date: 2025-01-15
Receipt Amount: 95.50
Diagnosis: Sinus infection
GST Inclusive: Yes

If you want to do anything else with these claims, please let me know.

Final answer: Here are your medical claims:

Claim ID: claim-95302
Status: submitted
Medical Provider: ABC Medical Clinic
Receipt No: R-12345
Receipt Date: 2025-01-15
Receipt Amount: 95.50
Diagnosis: Sinus infection
GST Inclusive: Yes

If you want to do anything else with these claims, please let me know.

[Step 2: Duration 1.78 seconds| Input tokens: 4,770 | Output tokens: 114]

Agent Response:
Here are your medical claims:

Claim ID: claim-95302
Status: submitted
Medical Provider: ABC Medical Clinic
Receipt No: R-12345
Receipt Date: 2025-01-15
Receipt Amount: 95.50
Diagnosis: Sinus infection
GST Inclusive: Yes

If you want to do anything else with these claims, please let me know.



In [20]:
# Test 12: Agent - Create a new draft (simulating receipt extraction)
# Note: The agent should ask for confirmation, but we'll simulate the full flow
receipt_text = """
I have a medical receipt from XYZ Hospital.
Receipt number: R-54321
Date: 2025-01-20
Amount: $120.00
Diagnosis: Annual checkup
GST is included in the amount.
"""

response = claims_smol_agent.run(
    f"I have a medical receipt. Here are the details:\n{receipt_text}\n\n"
    "Please create a draft claim with these details. I confirm they are correct."
)
print("Agent Response (Draft Creation):")
print(response)
print("\n" + "="*80)


╭────────────────────────────────────────── New run - claims_specialist ──────────────────────────────────────────╮
│                                                                                                                 │
│ I have a medical receipt. Here are the details:                                                                 │
│                                                                                                                 │
│ I have a medical receipt from XYZ Hospital.                                                                     │
│ Receipt number: R-54321                                                                                         │
│ Date: 2025-01-20                                                                                                │
│ Amount: $120.00                                                                                                 │
│ Diagnosis: Annual checkup                                                                                       │
│ GST is included in the amount.                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
│ Please create a draft claim with these details. I confirm they are correct.                                     │
│                                                                                                                 │
╰─ OpenAIModel - gpt-4.1-mini ────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'draft_medical_claim_tool' with arguments: {'employee_id': 'mark_tan', 'medical_provider': 'XYZ   │
│ Hospital', 'receipt_no': 'R-54321', 'receipt_date': '2025-01-20', 'receipt_amount': 120, 'diagnosis': 'Annual   │
│ checkup', 'gst_inclusive': True}                                                                                │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: {"status": "draft", "message": "Draft created successfully.", "claim_id": "claim-92058", "details": 
{"medical_provider": "XYZ Hospital", "receipt_no": "R-54321", "receipt_date": "2025-01-20", "receipt_amount": 
120.0, "diagnosis": "Annual checkup", "gst_inclusive": true}}

[Step 1: Duration 1.48 seconds| Input tokens: 2,363 | Output tokens: 63]

...

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'final_answer' with arguments: {'answer': 'The draft medical claim has been successfully created  │
│ with the following details:\n- Medical Provider: XYZ Hospital\n- Receipt Number: R-54321\n- Receipt Date:       │
│ 2025-01-20\n- Amount: $120.00\n- Diagnosis: Annual checkup\n- GST Inclusive: Yes\n\nIf you would like to submit │
│ this claim or make any updates, please let me know.'}                                                           │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: The draft medical claim has been successfully created with the following details:
- Medical Provider: XYZ Hospital
- Receipt Number: R-54321
- Receipt Date: 2025-01-20
- Amount: $120.00
- Diagnosis: Annual checkup
- GST Inclusive: Yes

If you would like to submit this claim or make any updates, please let me know.

Final answer: The draft medical claim has been successfully created with the following details:
- Medical Provider: XYZ Hospital
- Receipt Number: R-54321
- Receipt Date: 2025-01-20
- Amount: $120.00
- Diagnosis: Annual checkup
- GST Inclusive: Yes

If you would like to submit this claim or make any updates, please let me know.

[Step 2: Duration 1.79 seconds| Input tokens: 4,932 | Output tokens: 158]

Agent Response (Draft Creation):
The draft medical claim has been successfully created with the following details:
- Medical Provider: XYZ Hospital
- Receipt Number: R-54321
- Receipt Date: 2025-01-20
- Amount: $120.00
- Diagnosis: Annual checkup
- GST Inclusive: Yes

If you would like to submit this claim or make any updates, please let me know.



In [21]:
# Test 13: Agent - Update a draft claim
# First, let's create another draft to update
draft_result = ct.draft_medical_claim(
    employee_id=employee_id,
    medical_provider="Test Clinic",
    receipt_no="R-88888",
    receipt_date="2025-01-25",
    receipt_amount=75.00,
    diagnosis="Initial diagnosis",
    gst_inclusive=False
)
draft_data = json.loads(draft_result)
test_claim_id = draft_data.get("claim_id")

if test_claim_id:
    response = claims_smol_agent.run(
        f"I made a mistake in claim {test_claim_id}. "
        "The receipt amount should be $85.00 instead of $75.00. Can you update it?"
    )
    print("Agent Response (Update Claim):")
    print(response)
    print("\n" + "="*80)


╭────────────────────────────────────────── New run - claims_specialist ──────────────────────────────────────────╮
│                                                                                                                 │
│ I made a mistake in claim claim-91035. The receipt amount should be $85.00 instead of $75.00. Can you update    │
│ it?                                                                                                             │
│                                                                                                                 │
╰─ OpenAIModel - gpt-4.1-mini ────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'update_medical_claim_tool' with arguments: {'employee_id': 'mark_tan', 'claim_id':               │
│ 'claim-91035', 'field': 'receipt_amount', 'value': '85.00'}                                                     │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: {"status": "updated", "message": "Updated receipt_amount to 85.00 for claim claim-91035.", 
"current_data": {"medical_provider": "Test Clinic", "receipt_no": "R-88888", "receipt_date": "2025-01-25", 
"receipt_amount": 85.0, "diagnosis": "Initial diagnosis", "gst_inclusive": false}}

[Step 1: Duration 1.35 seconds| Input tokens: 2,322 | Output tokens: 39]

...

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'final_answer' with arguments: {'answer': 'The receipt amount for claim claim-91035 has been      │
│ successfully updated to $85.00.'}                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: The receipt amount for claim claim-91035 has been successfully updated to $85.00.

Final answer: The receipt amount for claim claim-91035 has been successfully updated to $85.00.

[Step 2: Duration 1.24 seconds| Input tokens: 4,823 | Output tokens: 71]

Agent Response (Update Claim):
The receipt amount for claim claim-91035 has been successfully updated to $85.00.



In [22]:
# Test 14: Agent - Submit a claim
if test_claim_id:
    response = claims_smol_agent.run(
        f"Yes, please submit claim {test_claim_id} for processing."
    )
    print("Agent Response (Submit Claim):")
    print(response)
    print("\n" + "="*80)


╭────────────────────────────────────────── New run - claims_specialist ──────────────────────────────────────────╮
│                                                                                                                 │
│ Yes, please submit claim claim-91035 for processing.                                                            │
│                                                                                                                 │
╰─ OpenAIModel - gpt-4.1-mini ────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'submit_medical_claim_tool' with arguments: {'employee_id': 'mark_tan', 'claim_id':               │
│ 'claim-91035'}                                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

--- SYSTEM: Claim claim-91035 formally submitted to backend ---


Observations: {"status": "submitted", "message": "Claim claim-91035 has been submitted for processing.", 
"employee_id": "mark_tan"}

...

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'final_answer' with arguments: {'answer': 'The claim claim-91035 has been successfully submitted  │
│ for processing.'}                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: The claim claim-91035 has been successfully submitted for processing.

Final answer: The claim claim-91035 has been successfully submitted for processing.

[Step 2: Duration 0.68 seconds| Input tokens: 4,713 | Output tokens: 54]

Agent Response (Submit Claim):
The claim claim-91035 has been successfully submitted for processing.



In [23]:
# Test 15: Agent - Full workflow simulation
print("="*80)
print("FULL WORKFLOW TEST")
print("="*80)

# Step 1: User provides receipt information
workflow_response = claims_smol_agent.run(
    "I have a medical receipt from 'City Medical Center'. "
    "Receipt number is R-77777, dated 2025-02-01, amount is $150.00. "
    "It was for a flu vaccination. GST is included. "
    "Please create a draft claim - I confirm all details are correct."
)
print("\n[Step 1] Draft Creation:")
print(workflow_response)

# Extract claim_id from response (if present)
import re
claim_match = re.search(r'claim-\d+', workflow_response)
if claim_match:
    workflow_claim_id = claim_match.group()
    print(f"\n✓ Extracted claim_id: {workflow_claim_id}")
    
    # Step 2: User wants to correct something
    workflow_response = claims_smol_agent.run(
        f"Actually, the amount for {workflow_claim_id} should be $145.00, not $150.00. "
        "Can you fix that?"
    )
    print("\n[Step 2] Update Claim:")
    print(workflow_response)
    
    # Step 3: User submits
    workflow_response = claims_smol_agent.run(
        f"Yes, please submit {workflow_claim_id} now."
    )
    print("\n[Step 3] Submit Claim:")
    print(workflow_response)
else:
    print("\n✗ Could not extract claim_id from response")


FULL WORKFLOW TEST


╭────────────────────────────────────────── New run - claims_specialist ──────────────────────────────────────────╮
│                                                                                                                 │
│ I have a medical receipt from 'City Medical Center'. Receipt number is R-77777, dated 2025-02-01, amount is     │
│ $150.00. It was for a flu vaccination. GST is included. Please create a draft claim - I confirm all details are │
│ correct.                                                                                                        │
│                                                                                                                 │
╰─ OpenAIModel - gpt-4.1-mini ────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'draft_medical_claim_tool' with arguments: {'employee_id': 'mark_tan', 'medical_provider': 'City  │
│ Medical Center', 'receipt_no': 'R-77777', 'receipt_date': '2025-02-01', 'receipt_amount': 150, 'diagnosis':     │
│ 'flu vaccination', 'gst_inclusive': True}                                                                       │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: {"status": "draft", "message": "Draft created successfully.", "claim_id": "claim-69694", "details": 
{"medical_provider": "City Medical Center", "receipt_no": "R-77777", "receipt_date": "2025-02-01", 
"receipt_amount": 150.0, "diagnosis": "flu vaccination", "gst_inclusive": true}}

...

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'final_answer' with arguments: {'answer': 'The claim with ID claim-69694 has been successfully    │
│ submitted for processing.'}                                                                                     │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: The claim with ID claim-69694 has been successfully submitted for processing.

Final answer: The claim with ID claim-69694 has been successfully submitted for processing.

[Step 2: Duration 0.69 seconds| Input tokens: 4,712 | Output tokens: 56]


[Step 3] Submit Claim:
The claim with ID claim-69694 has been successfully submitted for processing.


## 3. Error Handling Tests

Test how the agent handles various error scenarios.


In [24]:
# Test 16: Agent - Invalid date format
error_response = claims_smol_agent.run(
    "I have a receipt dated 01/15/2025 from ABC Clinic, receipt R-11111, "
    "amount $50.00, diagnosis: test. GST included. Please create a draft."
)
print("Agent Response (Invalid Date):")
print(error_response)
print("\n" + "="*80)


╭────────────────────────────────────────── New run - claims_specialist ──────────────────────────────────────────╮
│                                                                                                                 │
│ I have a receipt dated 01/15/2025 from ABC Clinic, receipt R-11111, amount $50.00, diagnosis: test. GST         │
│ included. Please create a draft.                                                                                │
│                                                                                                                 │
╰─ OpenAIModel - gpt-4.1-mini ────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'draft_medical_claim_tool' with arguments: {'employee_id': 'mark_tan', 'medical_provider': 'ABC   │
│ Clinic', 'receipt_no': 'R-11111', 'receipt_date': '2025-01-15', 'receipt_amount': 50, 'diagnosis': 'test',      │
│ 'gst_inclusive': True}                                                                                          │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: {"status": "draft", "message": "Draft created successfully.", "claim_id": "claim-86770", "details": 
{"medical_provider": "ABC Clinic", "receipt_no": "R-11111", "receipt_date": "2025-01-15", "receipt_amount": 50.0, 
"diagnosis": "test", "gst_inclusive": true}}

[Step 1: Duration 0.92 seconds| Input tokens: 2,330 | Output tokens: 61]

...

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'final_answer' with arguments: {'answer': 'Draft medical claim has been created successfully with │
│ the following details:\n- Medical Provider: ABC Clinic\n- Receipt Number: R-11111\n- Receipt Date:              │
│ 2025-01-15\n- Receipt Amount: $50.00\n- Diagnosis: test\n- GST Inclusive: Yes\nClaim ID: claim-86770\nPlease    │
│ review and let me know if you want to make any changes or submit this claim.'}                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: Draft medical claim has been created successfully with the following details:
- Medical Provider: ABC Clinic
- Receipt Number: R-11111
- Receipt Date: 2025-01-15
- Receipt Amount: $50.00
- Diagnosis: test
- GST Inclusive: Yes
Claim ID: claim-86770
Please review and let me know if you want to make any changes or submit this claim.

Final answer: Draft medical claim has been created successfully with the following details:
- Medical Provider: ABC Clinic
- Receipt Number: R-11111
- Receipt Date: 2025-01-15
- Receipt Amount: $50.00
- Diagnosis: test
- GST Inclusive: Yes
Claim ID: claim-86770
Please review and let me know if you want to make any changes or submit this claim.

[Step 2: Duration 1.28 seconds| Input tokens: 4,860 | Output tokens: 161]

Agent Response (Invalid Date):
Draft medical claim has been created successfully with the following details:
- Medical Provider: ABC Clinic
- Receipt Number: R-11111
- Receipt Date: 2025-01-15
- Receipt Amount: $50.00
- Diagnosis: test
- GST Inclusive: Yes
Claim ID: claim-86770
Please review and let me know if you want to make any changes or submit this claim.



In [25]:
# Test 17: Agent - Try to update non-existent claim
error_response = claims_smol_agent.run(
    "Can you update claim claim-99999? Change the amount to $200.00."
)
print("Agent Response (Non-existent Claim):")
print(error_response)
print("\n" + "="*80)


╭────────────────────────────────────────── New run - claims_specialist ──────────────────────────────────────────╮
│                                                                                                                 │
│ Can you update claim claim-99999? Change the amount to $200.00.                                                 │
│                                                                                                                 │
╰─ OpenAIModel - gpt-4.1-mini ────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'update_medical_claim_tool' with arguments: {'employee_id': 'mark_tan', 'claim_id':               │
│ 'claim-99999', 'field': 'receipt_amount', 'value': '200.00'}                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: {"error": "Claim not found."}

[Step 1: Duration 0.79 seconds| Input tokens: 2,308 | Output tokens: 39]

...

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'final_answer' with arguments: {'answer': 'The claim with ID claim-99999 was not found. Please    │
│ check the claim ID and provide the correct one if you want to update the amount.'}                              │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: The claim with ID claim-99999 was not found. Please check the claim ID and provide the correct one if
you want to update the amount.

Final answer: The claim with ID claim-99999 was not found. Please check the claim ID and provide the correct one if
you want to update the amount.

[Step 2: Duration 0.74 seconds| Input tokens: 4,718 | Output tokens: 82]

Agent Response (Non-existent Claim):
The claim with ID claim-99999 was not found. Please check the claim ID and provide the correct one if you want to update the amount.



## 4. Summary and Verification

Final verification that all claims are properly stored.


In [26]:
# Final verification: List all claims
final_result = ct.list_claims(employee_id)
final_data = json.loads(final_result)

print("="*80)
print("FINAL CLAIMS SUMMARY")
print("="*80)
print(f"Employee ID: {final_data.get('employee_id')}")
print(f"Total Claims: {len(final_data.get('claims', []))}")
print("\nAll Claims:")
print(json.dumps(final_data, indent=2))


FINAL CLAIMS SUMMARY
Employee ID: mark_tan
Total Claims: 5

All Claims:
{
  "employee_id": "mark_tan",
  "claims": [
    {
      "claim_id": "claim-95302",
      "status": "submitted",
      "medical_provider": "ABC Medical Clinic",
      "receipt_no": "R-12345",
      "receipt_date": "2025-01-15",
      "receipt_amount": 95.5,
      "diagnosis": "Sinus infection",
      "gst_inclusive": true
    },
    {
      "claim_id": "claim-92058",
      "status": "draft",
      "medical_provider": "XYZ Hospital",
      "receipt_no": "R-54321",
      "receipt_date": "2025-01-20",
      "receipt_amount": 120.0,
      "diagnosis": "Annual checkup",
      "gst_inclusive": true
    },
    {
      "claim_id": "claim-91035",
      "status": "submitted",
      "medical_provider": "Test Clinic",
      "receipt_no": "R-88888",
      "receipt_date": "2025-01-25",
      "receipt_amount": 85.0,
      "diagnosis": "Initial diagnosis",
      "gst_inclusive": false
    },
    {
      "claim_id": "claim-69694",


## 5. Cleanup (Optional)

Uncomment the cell below to clean up test data from the database.


In [27]:
# Optional: Clean up test data
# Uncomment to delete all claims for the test employee

# import sqlite3
# conn = sqlite3.connect('claims_poc.db')
# conn.execute("DELETE FROM claims WHERE employee_id = ?", (employee_id,))
# conn.commit()
# conn.close()
# print(f"✓ Cleaned up all claims for {employee_id}")
